In [1]:
import os

REPO = 'giomamaca/walmart-sales-forecasting'

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    from google.colab import userdata, files

    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
        print('Upload your kaggle.json')
        uploaded = files.upload()
        shutil.move('kaggle.json', f'{kaggle_dir}/kaggle.json')
        os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)

    github_token = userdata.get('GITHUB_TOKEN')
    repo_dir = f'/content/{REPO.split("/")[1]}'
    if not os.path.exists(repo_dir):
        !git clone -q https://{github_token}@github.com/{REPO}.git {repo_dir}
    os.chdir(repo_dir)

    !pip install -q kaggle mlflow wandb lightgbm

    import wandb
    wandb.login(key=userdata.get('WANDB_API_KEY'))

    !kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p data
    !unzip -o -q data/walmart-recruiting-store-sales-forecasting.zip -d data
    !for f in data/*.zip; do unzip -o -q "$f" -d data; done

    !mlflow db upgrade sqlite:///mlflow.db

    DATA_DIR = 'data'
else:
    DATA_DIR = '..'

Upload your kaggle.json


Saving kaggle.json to kaggle.json
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adola23 (adola23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100% 2.70M/2.70M [00:00<00:00, 64.7MB/s]

2026/07/11 10:42:00 INFO mlflow.store.db.utils: Updating database tables


In [2]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import wandb
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

FIXED = dict(verbose=-1)

In [3]:
train = pd.read_csv(f'{DATA_DIR}/train.csv', parse_dates=['Date'])
features = pd.read_csv(f'{DATA_DIR}/features.csv', parse_dates=['Date'])
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')

train.shape, features.shape, stores.shape

((421570, 5), (8190, 12), (45, 3))

In [4]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.average(np.abs(y_true - y_pred), weights=weights)

In [5]:
MARKDOWN_COLS = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
PRE_CHRISTMAS = pd.to_datetime(['2010-12-24', '2011-12-23', '2012-12-21'])

FEATURE_COLS = [
    'Store', 'Dept', 'IsHoliday', 'Size', 'Type',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'IsPreChristmas',
    'Sales_Lag52', 'Sales_CalLag1y', 'Sales_CalLag2y', 'Sales_CalLag_Mean',
    'Series_Mean', 'Series_Std', 'Series_WoyMean',
    'Dept_WoyScaled', 'Dept_Mean', 'Store_Mean',
]

# v5: leakage fix. v3/v4-shi agregatebi (Series_Mean, Series_WoyMean da a.sh.)
# fit-is dros TREN-is yvela striqonze itvleboda, anu striqonis sakutari
# gayidvebi sakutar feature-shi ijda (woy agregatshi es 33-50%-ia!). modeli
# amit "swavlobda" pasuxis nawilobriv gadaweras da testze ingreoda.
# gamosworeba: leave-one-out. tu striqoni fit-is monacemebshi iyo, agregats
# aklem misi sakutari wvlili ((sum - own) / (n - 1)). testis striqonebistvis
# (romlebic fit-shi ar iyo) sruli agregati rcheba. as train-is da test-is
# feature-ebs zustad erti da igive semantika aqvt.
class FeatureBuilderV5(BaseEstimator, TransformerMixin):
    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df
        self.features_df = features_df

    def fit(self, X, y=None):
        df = X[['Store', 'Dept', 'Date']].copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df['y'] = (y.values if hasattr(y, 'values') else y).astype(float)

        self.history_ = df.set_index(['Store', 'Dept', 'Date'])['y']

        g = df.groupby(['Store', 'Dept'])['y']
        self.series_sum_ = g.sum()
        self.series_sumsq_ = (df['y'] ** 2).groupby([df['Store'], df['Dept']]).sum()
        self.series_cnt_ = g.count()

        woy = df['Date'].dt.isocalendar().week.astype(int)
        gw = df['y'].groupby([df['Store'], df['Dept'], woy])
        self.woy_sum_ = gw.sum()
        self.woy_cnt_ = gw.count()

        series_mean_full = self.series_sum_ / self.series_cnt_
        sm = df.set_index(['Store', 'Dept']).index.map(series_mean_full)
        norm = pd.Series(np.where(sm > 0, df['y'] / sm, np.nan), index=df.index)
        self.norm_history_ = pd.Series(norm.values,
                                       index=pd.MultiIndex.from_arrays([df['Store'], df['Dept'], df['Date']]))
        gs = norm.groupby([df['Dept'], woy])
        self.shape_sum_ = gs.sum()
        self.shape_cnt_ = gs.count()

        self.dept_mean_ = df.groupby('Dept')['y'].mean()
        self.store_mean_ = df.groupby('Store')['y'].mean()
        return self

    def _week_lag(self, df, weeks):
        idx = pd.MultiIndex.from_arrays(
            [df['Store'], df['Dept'], df['Date'] - pd.Timedelta(weeks=weeks)])
        return self.history_.reindex(idx).values

    def _cal_lag(self, df, years):
        pairs = {}
        for d in df['Date'].unique():
            d = pd.Timestamp(d)
            wts = {}
            for i in range(7):
                anniv = (d - pd.Timedelta(days=i)) - pd.DateOffset(years=years)
                we = anniv + pd.Timedelta(days=(4 - anniv.weekday()) % 7)
                wts[we] = wts.get(we, 0.0) + 1.0 / 7.0
            pairs[d] = sorted(wts.items())

        num = np.zeros(len(df))
        eff = np.zeros(len(df))
        max_p = max(len(v) for v in pairs.values())
        for j in range(max_p):
            we_col = df['Date'].map(lambda d, j=j: pairs[d][j][0] if len(pairs[d]) > j else pd.NaT)
            wt_col = df['Date'].map(lambda d, j=j: pairs[d][j][1] if len(pairs[d]) > j else 0.0).values
            idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], we_col])
            vals = self.history_.reindex(idx).values.astype(float)
            ok = ~np.isnan(vals)
            num[ok] += wt_col[ok] * vals[ok]
            eff[ok] += wt_col[ok]
        return np.where(eff > 0, num / np.maximum(eff, 1e-9), np.nan)

    def transform(self, X):
        df = X.copy()
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.merge(self.stores_df, on='Store', how='left')
        df = df.merge(self.features_df.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

        df[MARKDOWN_COLS] = df[MARKDOWN_COLS].fillna(0)
        df['Type'] = df['Type'].map({'A': 0, 'B': 1, 'C': 2})
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['IsPreChristmas'] = df['Date'].isin(PRE_CHRISTMAS).astype(int)

        df['Sales_Lag52'] = self._week_lag(df, 52)
        df['Sales_CalLag1y'] = self._cal_lag(df, 1)
        df['Sales_CalLag2y'] = self._cal_lag(df, 2)
        df['Sales_CalLag_Mean'] = pd.concat(
            [df['Sales_CalLag1y'], df['Sales_CalLag2y']], axis=1).mean(axis=1).values

        row_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], df['Date']])
        own = self.history_.reindex(row_idx).values.astype(float)
        seen = ~np.isnan(own)
        own0 = np.where(seen, own, 0.0)

        pair_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept']])
        s = self.series_sum_.reindex(pair_idx).values.astype(float)
        ss = self.series_sumsq_.reindex(pair_idx).values.astype(float)
        n = self.series_cnt_.reindex(pair_idx).values.astype(float)

        n_eff = n - seen
        with np.errstate(invalid='ignore', divide='ignore'):
            mean = np.where(n_eff > 0, (s - own0) / n_eff, np.nan)
            var = np.where(n_eff > 0, (ss - own0 ** 2) / n_eff - mean ** 2, np.nan)
        df['Series_Mean'] = mean
        df['Series_Std'] = np.sqrt(np.clip(var, 0, None))

        woy_t = df['Date'].dt.isocalendar().week.astype(int)
        woy_idx = pd.MultiIndex.from_arrays([df['Store'], df['Dept'], woy_t])
        ws = self.woy_sum_.reindex(woy_idx).values.astype(float)
        wn = self.woy_cnt_.reindex(woy_idx).values.astype(float)
        wn_eff = wn - seen
        with np.errstate(invalid='ignore', divide='ignore'):
            df['Series_WoyMean'] = np.where(wn_eff > 0, (ws - own0) / wn_eff, np.nan)

        own_norm = self.norm_history_.reindex(row_idx).values.astype(float)
        own_norm0 = np.where(np.isnan(own_norm), 0.0, own_norm)
        seen_norm = seen & ~np.isnan(own_norm)
        shape_idx = pd.MultiIndex.from_arrays([df['Dept'], woy_t])
        hs = self.shape_sum_.reindex(shape_idx).values.astype(float)
        hn = self.shape_cnt_.reindex(shape_idx).values.astype(float)
        hn_eff = hn - seen_norm
        with np.errstate(invalid='ignore', divide='ignore'):
            shape = np.where(hn_eff > 0, (hs - own_norm0) / hn_eff, np.nan)
        df['Dept_WoyScaled'] = shape * df['Series_Mean'].values

        df['Dept_Mean'] = self.dept_mean_.reindex(df['Dept']).values
        df['Store_Mean'] = self.store_mean_.reindex(df['Store']).values

        return df[FEATURE_COLS]

In [6]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('LightGBM_v5_Training')

2026/07/11 10:42:12 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_v5_Training' does not exist. Creating a new experiment.


<Experiment: artifact_location='/content/walmart-sales-forecasting/mlruns/11', creation_time=1783766532458, effective_trace_archival_retention=None, experiment_id='11', last_update_time=1783766532458, lifecycle_stage='active', name='LightGBM_v5_Training', tags={}, trace_location=None, workspace='default'>

In [7]:
with mlflow.start_run(run_name='LightGBM_v5_Cleaning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v5_Cleaning', reinit='finish_previous')

    n_negative = int((train['Weekly_Sales'] < 0).sum())
    missing_markdown_pct = float(features[MARKDOWN_COLS].isna().mean().mean())

    mlflow.log_param('negative_sales_rows', n_negative)
    mlflow.log_param('negative_sales_action', 'clip_to_zero')
    mlflow.log_metric('missing_markdown_pct', missing_markdown_pct)
    wandb.log({'negative_sales_rows': n_negative, 'missing_markdown_pct': missing_markdown_pct})
    wandb.finish()

    y_train = train['Weekly_Sales'].clip(lower=0)

missing_markdown_pct,▁
negative_sales_rows,▁
missing_markdown_pct,0.55849
negative_sales_rows,1285


In [8]:
def time_based_splits(dates, n_splits=3):
    dates = np.sort(dates.unique())
    fold_size = len(dates) // (n_splits + 1)
    splits = []
    for i in range(1, n_splits + 1):
        train_end = dates[fold_size * i - 1]
        val_end = dates[min(fold_size * (i + 1) - 1, len(dates) - 1)]
        splits.append((train_end, val_end))
    return splits

splits = time_based_splits(train['Date'])
splits

# hiperparametrebs ori test-formis holdout-ის sashualoze vircheavt (ertze
# shercheva ubralod im fanjris gamartlebas poulobs). orive igive formisaa
# rac kaggle-is testi: oqtombramde vswavlobt, shemdegi noembridan ivlisamde vafasebt
HOLDOUTS = [
    (pd.Timestamp('2010-10-29'), pd.Timestamp('2011-07-29')),
    (pd.Timestamp('2011-10-28'), pd.Timestamp('2012-07-27')),
]

In [9]:
SEARCH_SPACE = {
    'n_estimators': [500, 1000, 1500],
    'num_leaves': [31, 63, 127, 255],
    'learning_rate': [0.03, 0.05, 0.1],
    'min_child_samples': [20, 50, 100, 200],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
}
N_TRIALS = 20

rng = np.random.default_rng(42)

def sample_params(rng):
    return {k: v[rng.integers(len(v))] for k, v in SEARCH_SPACE.items()}

holdout_data = []
for cutoff, end in HOLDOUTS:
    tm = train['Date'] <= cutoff
    vm = (train['Date'] > cutoff) & (train['Date'] <= end)
    b = FeatureBuilderV5(stores, features)
    b.fit(train.loc[tm, ['Store', 'Dept', 'Date']], y_train[tm])
    holdout_data.append((
        b.transform(train[tm]), y_train[tm],
        b.transform(train[vm]), y_train[vm],
        train.loc[vm, 'IsHoliday'].values,
    ))

def holdout_scores(params):
    scores = []
    for X_tr, y_tr, X_val, y_val, hol in holdout_data:
        model = LGBMRegressor(**params, **FIXED)
        model.fit(X_tr, y_tr)
        scores.append(wmae(y_val.values, model.predict(X_val), hol))
    return scores

with mlflow.start_run(run_name='LightGBM_v5_Tuning'):
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v5_Tuning', reinit='finish_previous')

    best_score, best_params, best_pair = None, None, None
    for t in range(N_TRIALS):
        params = sample_params(rng)
        scores = holdout_scores(params)
        mean = float(np.mean(scores))
        mlflow.log_metric('trial_wmae_holdout_mean', mean, step=t)
        wandb.log({'trial': t, 'trial_wmae_holdout_mean': mean,
                   'wmae_h2010': scores[0], 'wmae_h2011': scores[1], **params})
        if best_score is None or mean < best_score:
            best_score, best_params, best_pair = mean, params, scores
        print(t, round(mean, 1), [round(s, 1) for s in scores], params)

    mlflow.log_metric('best_wmae_holdout_mean', best_score)
    mlflow.log_metric('best_wmae_holdout_2011', best_pair[1])
    mlflow.log_dict(best_params, 'best_params.json')
    wandb.log({'best_wmae_holdout_mean': best_score})
    wandb.finish()

best_score, best_pair, best_params

0 3368.2 [np.float64(4797.7), np.float64(1938.7)] {'n_estimators': 500, 'num_leaves': 255, 'learning_rate': 0.05, 'min_child_samples': 50, 'subsample': 0.8, 'colsample_bytree': 1.0}
1 3495.0 [np.float64(5009.9), np.float64(1980.2)] {'n_estimators': 500, 'num_leaves': 127, 'learning_rate': 0.03, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 1.0}
2 3396.1 [np.float64(4818.8), np.float64(1973.4)] {'n_estimators': 1500, 'num_leaves': 255, 'learning_rate': 0.1, 'min_child_samples': 200, 'subsample': 0.8, 'colsample_bytree': 0.7}
3 3270.7 [np.float64(4596.2), np.float64(1945.2)] {'n_estimators': 1500, 'num_leaves': 63, 'learning_rate': 0.05, 'min_child_samples': 50, 'subsample': 0.7, 'colsample_bytree': 1.0}
4 3363.1 [np.float64(4777.6), np.float64(1948.6)] {'n_estimators': 1500, 'num_leaves': 127, 'learning_rate': 0.05, 'min_child_samples': 200, 'subsample': 0.8, 'colsample_bytree': 0.8}
5 3277.4 [np.float64(4519.9), np.float64(2034.8)] {'n_estimators': 1000, 'num_leaves': 

best_wmae_holdout_mean,▁
colsample_bytree,██▁█▃▁██▃█▁▁▁██▁▁▃▃▁
learning_rate,▃▁█▃▃▁▁▁█▃▃▁██▁█▃█▁▁
min_child_samples,▂▁█▂█▄▄█▁▁█▂▂▂▂▂▁██▁
n_estimators,▁▁███▅███▅██▅▅▅▅▁█▅▅
num_leaves,█▄█▂▄▁█▂█▁▄▂▁▄▄█▄▁▄▄
subsample,▃▃▃▁▃█▁▃▃█▃█▁▁████▃▁
trial,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
trial_wmae_holdout_mean,▆▇▆▅▆▅▅▅▅█▅▂▂▅▅▁▃▅▅▃
wmae_h2010,▆▇▆▅▆▄▆▅▆█▅▂▂▅▅▁▃▅▅▄
+1,...


(2980.489864725829,
 [np.float64(4025.267492529285), np.float64(1935.7122369223732)],
 {'n_estimators': 1000,
  'num_leaves': 255,
  'learning_rate': 0.1,
  'min_child_samples': 50,
  'subsample': 1.0,
  'colsample_bytree': 0.7})

In [10]:
with mlflow.start_run(run_name='LightGBM_v5_CV'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v5_CV', config=best_params, reinit='finish_previous')

    fold_scores = []
    for i, (train_end, val_end) in enumerate(splits):
        ftm = train['Date'] <= train_end
        fvm = (train['Date'] > train_end) & (train['Date'] <= val_end)

        pipe = Pipeline([
            ('features', FeatureBuilderV5(stores, features)),
            ('model', LGBMRegressor(**best_params, **FIXED)),
        ])
        pipe.fit(train.loc[ftm, ['Store', 'Dept', 'Date', 'IsHoliday']], y_train[ftm])
        preds = pipe.predict(train.loc[fvm, ['Store', 'Dept', 'Date', 'IsHoliday']])

        score = wmae(y_train[fvm].values, preds, train.loc[fvm, 'IsHoliday'].values)
        fold_scores.append(score)
        mlflow.log_metric('wmae', score, step=i)
        wandb.log({'fold': i, 'wmae': score})

    mlflow.log_metric('wmae_mean', float(np.mean(fold_scores)))
    mlflow.log_metric('wmae_std', float(np.std(fold_scores)))
    mlflow.log_metric('wmae_holdout', best_pair[1])
    mlflow.log_metric('wmae_holdout_mean', best_score)
    wandb.log({'wmae_mean': float(np.mean(fold_scores)), 'wmae_std': float(np.std(fold_scores)),
               'wmae_holdout': best_pair[1], 'wmae_holdout_mean': best_score})
    wandb.finish()

fold_scores

fold,▁▅█
wmae,█▂▁
wmae_holdout,▁
wmae_holdout_mean,▁
wmae_mean,▁
wmae_std,▁
fold,2
wmae,1600.86104
wmae_holdout,1935.71224
wmae_holdout_mean,2980.48986
wmae_mean,2720.91628


[np.float64(4588.684165743944),
 np.float64(1973.2036447462497),
 np.float64(1600.861036588524)]

In [11]:
with mlflow.start_run(run_name='LightGBM_v5_Final'):
    mlflow.log_params(best_params)
    wandb.init(project='walmart-sales-forecasting', name='LightGBM_v5_Final', config=best_params, reinit='finish_previous')

    pipeline = Pipeline([
        ('features', FeatureBuilderV5(stores, features)),
        ('model', LGBMRegressor(**best_params, **FIXED)),
    ])
    pipeline.fit(train[['Store', 'Dept', 'Date', 'IsHoliday']], y_train)

    preds = pipeline.predict(train[['Store', 'Dept', 'Date', 'IsHoliday']])
    train_wmae = wmae(y_train.values, preds, train['IsHoliday'].values)
    mlflow.log_metric('train_wmae', train_wmae)
    wandb.log({'train_wmae': train_wmae})

    mlflow.sklearn.log_model(pipeline, name='model', serialization_format='cloudpickle')
    wandb.finish()

train_wmae

2026/07/11 11:26:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


train_wmae,▁
train_wmae,691.93665


np.float64(691.9366477817906)

In [12]:
if IN_COLAB:
    import requests
    gh_user = requests.get('https://api.github.com/user', headers={'Authorization': f'token {github_token}'}).json()['login']
    !git config user.name "{gh_user}"
    !git config user.email "{gh_user}@users.noreply.github.com"

    !git add mlflow.db mlruns
    !git commit -m "Run model_experiment_LightGBM_v5.ipynb in Colab"
    !git push

[main 3debf24] Run model_experiment_LightGBM_v5.ipynb in Colab
 7 files changed, 83 insertions(+)
 create mode 100644 mlruns/11/f6e3da1135214fd68f32f66c52bf8c67/artifacts/best_params.json
 create mode 100644 mlruns/11/models/m-446ca6517a3940a4bcb998f1672dd736/artifacts/MLmodel
 create mode 100644 mlruns/11/models/m-446ca6517a3940a4bcb998f1672dd736/artifacts/conda.yaml
 create mode 100644 mlruns/11/models/m-446ca6517a3940a4bcb998f1672dd736/artifacts/model.pkl
 create mode 100644 mlruns/11/models/m-446ca6517a3940a4bcb998f1672dd736/artifacts/python_env.yaml
 create mode 100644 mlruns/11/models/m-446ca6517a3940a4bcb998f1672dd736/artifacts/requirements.txt
Enumerating objects: 19, done.
Counting objects: 100% (19/19), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (16/16), 13.62 MiB | 3.18 MiB/s, done.
Total 16 (delta 3), reused 3 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To ht